# 02 — All-available matched-cohort age-head anatomic explainability

## Quick start

Run notebook 01 through section 10 first. This notebook loads every eligible
race-specific cohort and its separately age/sex/comorbidity-matched White
cohort. It explains the frozen global head, the all-available target-group head,
and that comparison's matched-White head on the same target images. It also
compares the target population's own-head maps with matched White participants'
own-head maps.

Processing is resumable in batches of four comparison-participant records.
Durable output is written to
`16_algorithm_fairness/07_all_available_matched_anatomic_explainability`.

The 1,024 coefficients are latent features, not named physiology. Spatial
interpretation uses exact patch contributions intersected with FR-U-Net vessel
segmentation and localized optic-disc, peripapillary, and foveal ROIs. The disc
and fovea are localized circular ROIs—not pixel segmentations.


In [ ]:
%pip install -q "numpy>=2.0,<2.3" "git+https://github.com/berenslab/fundus_image_toolbox.git@d7757e28fbf639856b53cfe00019f605af8c1f17" "huggingface_hub>=0.24" "timm>=0.9,<1.1

In [ ]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

In [ ]:
%pip install -q --no-deps "opencv-python-headless==4.11.0.86

In [ ]:
dbutils.library.restartPython()

## 1. Fixed analysis configuration

Only the Hugging Face credential is a widget. Paths and analytic settings are intentionally fixed so a restart cannot silently change the cohort or output location.


In [ ]:
from pathlib import Path
import gc, hashlib, importlib, json, math, os, re, shutil, sys, time, uuid

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage

cv2.setNumThreads(1)
if cv2.__version__ != "4.11.0":
    raise RuntimeError(f"Expected OpenCV 4.11.0; imported {cv2.__version__}. Rerun from the top.")
try:
    dbutils.widgets.get("hf_token")
except Exception:
    dbutils.widgets.text("hf_token", "", "Hugging Face token (temporary)")

repo_root = Path("/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina")
fairness_root = Path(
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/Age_Glaucoma/16_algorithm_fairness"
)
output_root = fairness_root / "07_all_available_matched_anatomic_explainability"
global_model_path = fairness_root / "03_age_model/CLSA_full_cohort_age_head.joblib"
matched_model_root = fairness_root / "03_age_model/race_matched_all_available_age_heads"
selection_path = fairness_root / "01_private/race_matched_all_available_image_selection_private.parquet"
fit_cache_dir = fairness_root.parent.parent / "model_checkpoints/fundus_image_toolbox"

segmentation_batch_size = 4
max_new_batches_per_run = 0
resume_batches = True
maximum_comparison_records = 0
vessel_threshold = 0.5
optic_disc_radius_scale = 0.20
fovea_radius_scale = 0.20
peripapillary_multiplier = 2.0
permutations = 5000
bootstrap_repetitions = 2000
registration_size = 256
review_examples_per_group = 2
device_requested = "auto"

for path in (global_model_path, selection_path, matched_model_root):
    if not path.exists():
        raise FileNotFoundError(f"Missing notebook 01 section-10 output: {path}")
module_root = repo_root / "src"
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))
import age_gap_extremes as _age_gap
import clsa_anatomic_explainability as _anatomy
import fundus_retfound_pipeline as _fundus
import retfound_age_anatomic_explainability as _age_xai
_age_gap = importlib.reload(_age_gap); _anatomy = importlib.reload(_anatomy)
_fundus = importlib.reload(_fundus); _age_xai = importlib.reload(_age_xai)
from age_gap_extremes import fundus_physiology_proxies
from clsa_anatomic_explainability import (
    attribution_region_metrics, build_anatomic_masks, disc_fovea_affine_matrix,
    participant_permutation_inference,
)
from fundus_retfound_pipeline import QualityConfig, RETFoundConfig, load_retfound_model, preprocess_fundus, write_frame, write_json
from retfound_age_anatomic_explainability import effective_linear_head, exact_multihead_patch_contributions, coefficient_similarity_table, paired_model_inference
output_root.mkdir(parents=True, exist_ok=True)
print("Output root:", output_root)


## 2. Load all comparison-specific cohorts and heads

The ledger may contain the same White participant in two independently matched
comparisons, so identity is enforced by `(comparison_group, participant_id)`.
Every record is explained only with three relevant heads: global, its target
group's head, and its comparison-specific matched-White head.


In [ ]:
def normalized_label(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).casefold())


selection = pd.read_parquet(selection_path)
required = {"comparison_group", "participant_id", "image_path", "age", "racial_background", "population_role", "match_role"}
missing = required - set(selection.columns)
if missing:
    raise ValueError(f"Expanded selection ledger is missing: {sorted(missing)}")
selection["participant_id"] = selection["participant_id"].astype(str)
selection["comparison_group"] = selection["comparison_group"].astype(str)
selection["age"] = pd.to_numeric(selection["age"], errors="coerce")
selection = selection.dropna(subset=["comparison_group", "participant_id", "image_path", "age"]).copy()
if selection.duplicated(["comparison_group", "participant_id"]).any():
    raise ValueError("Selection is not unique by comparison group and participant")
if maximum_comparison_records:
    selection = selection.groupby(["comparison_group", "population_role"], group_keys=False, sort=True).head(maximum_comparison_records)
if selection.empty:
    raise ValueError("No matched comparison records remain")

global_bundle = joblib.load(global_model_path)
model_bundles = {"global": global_bundle}
comparison_head_lookup = {}
model_metadata = [{"model_name": "global", "comparison_group": None, "model_role": "global", "path": str(global_model_path)}]
for model_path in sorted(matched_model_root.glob("*_age_head.joblib")):
    bundle = joblib.load(model_path)
    group = str(bundle.get("comparison_group", "")).strip()
    role = str(bundle.get("model_role", "")).strip()
    if not group or role not in {"target", "matched_white"}:
        continue
    model_name = f"comparison::{group}::{role}"
    if model_name in model_bundles:
        raise ValueError(f"Duplicate comparison head: {model_name}")
    model_bundles[model_name] = bundle
    comparison_head_lookup.setdefault(group, {})[role] = model_name
    model_metadata.append({"model_name": model_name, "comparison_group": group, "model_role": role, "path": str(model_path)})

comparison_groups = sorted(selection["comparison_group"].unique())
missing_heads = [group for group in comparison_groups if set(comparison_head_lookup.get(group, {})) != {"target", "matched_white"}]
if missing_heads:
    raise FileNotFoundError(f"Incomplete target/matched-White head pairs for: {missing_heads}")
linear_heads = {name: effective_linear_head(bundle) for name, bundle in model_bundles.items()}
if len({len(value[0]) for value in linear_heads.values()}) != 1:
    raise ValueError("Age-head embedding dimensions are inconsistent")

def relevant_model_names(comparison_group):
    pair = comparison_head_lookup[str(comparison_group)]
    return ["global", pair["target"], pair["matched_white"]]

display(selection.groupby(["comparison_group", "population_role"]).agg(participants=("participant_id", "nunique"), mean_age=("age", "mean")))
display(pd.DataFrame(model_metadata))


## 3. Latent coefficient audit

Coefficient similarity is reported globally and within every target-versus-
matched-White pair. Embedding dimensions remain unnamed latent features;
anatomic interpretation comes from the spatial analysis below.


In [ ]:
statistics_root = output_root / "04_statistics"
figure_root = output_root / "05_figures"
private_root = output_root / "01_private"
batch_root = output_root / "02_anatomic_batches"
artifact_root = output_root / "03_private_artifacts"
for path in (statistics_root, figure_root, private_root, batch_root, artifact_root):
    path.mkdir(parents=True, exist_ok=True)

coefficient_similarity = coefficient_similarity_table(linear_heads, reference="global", top_k=50)
write_frame(coefficient_similarity, statistics_root / "all_heads_vs_global_coefficient_similarity.csv")
pair_rows = []
top_rows = []
for group in comparison_groups:
    target_name = comparison_head_lookup[group]["target"]
    white_name = comparison_head_lookup[group]["matched_white"]
    target_coef = np.asarray(linear_heads[target_name][0]); white_coef = np.asarray(linear_heads[white_name][0])
    denominator = np.linalg.norm(target_coef) * np.linalg.norm(white_coef)
    top_target = set(np.argsort(np.abs(target_coef))[-50:]); top_white = set(np.argsort(np.abs(white_coef))[-50:])
    pair_rows.append({
        "comparison_group": group, "coefficient_cosine": float(target_coef @ white_coef / denominator),
        "coefficient_correlation": float(np.corrcoef(target_coef, white_coef)[0, 1]),
        "coefficient_sign_agreement": float(np.mean(np.sign(target_coef) == np.sign(white_coef))),
        "top_50_jaccard": float(len(top_target & top_white) / len(top_target | top_white)),
        "coefficient_l2_distance": float(np.linalg.norm(target_coef - white_coef)),
    })
for model_name, (coef, _) in linear_heads.items():
    for rank, dimension in enumerate(np.argsort(np.abs(coef))[::-1][:50], 1):
        top_rows.append({"model_name": model_name, "rank": rank, "embedding_dimension": int(dimension), "coefficient": float(coef[dimension]), "absolute_coefficient": float(abs(coef[dimension]))})
paired_coefficient_similarity = pd.DataFrame(pair_rows)
top_coefficients = pd.DataFrame(top_rows)
write_frame(paired_coefficient_similarity, statistics_root / "target_vs_matched_white_coefficient_similarity.csv")
write_frame(top_coefficients, statistics_root / "top_50_latent_coefficients_by_head.csv")
display(paired_coefficient_similarity.round(4))


## 4. Load RETFound and Fundus Image Toolbox models

FR-U-Net supplies pixel-level vessel masks. The landmark model supplies foveal and optic-disc centers, converted into fixed-radius ROIs using the same definitions as notebook 09.


In [ ]:
quality_config = QualityConfig(output_size=256, model_input_size=224, save_preprocessed=False)
if device_requested == "auto":
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
elif device_requested == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable")
    device = "cuda:0"
else:
    device = "cpu"

temporary_hf_token = dbutils.widgets.get("hf_token").strip()
if temporary_hf_token:
    os.environ["HF_TOKEN"] = temporary_hf_token
try:
    retfound_model, retfound_device, resolved_repo, resolved_checkpoint = load_retfound_model(
        RETFoundConfig(
            repo_path=None, checkpoint_path=None, allow_downloads=True,
            allow_repo_clone=True, device="cuda" if device.startswith("cuda") else "cpu", batch_size=1,
        )
    )
finally:
    os.environ.pop("HF_TOKEN", None)
    temporary_hf_token = ""

FIT_SOURCE_COMMIT = "d7757e28fbf639856b53cfe00019f605af8c1f17"
os.environ["FIT_CACHE_DIR"] = str(fit_cache_dir)
os.environ["TORCH_HOME"] = str(fit_cache_dir / "torch")
fit_cache_dir.mkdir(parents=True, exist_ok=True)
(fit_cache_dir / "torch").mkdir(parents=True, exist_ok=True)
import fundus_image_toolbox as fit
print("Loading fovea/optic-disc localizer...", flush=True)
landmark_model, landmark_checkpoint = fit.load_fovea_od_model(device=device, cache_dir=str(fit_cache_dir))
print("Loading vessel ensemble...", flush=True)
vessel_ensemble = fit.load_segmentation_ensemble(device=device, cache_dir=str(fit_cache_dir))
print("RETFound device/checkpoint:", retfound_device, resolved_checkpoint)
print("FIT device/version:", device, fit.__version__)


## 5. Resumable exact attribution and anatomic segmentation

Each image is encoded once, after which every linear head is decomposed exactly from the same patch tokens. A batch Parquet is committed only after all four images, attribution archives, and anatomy masks are complete. Re-running skips validated batches.


In [ ]:
local_root = Path(f"/local_disk0/tmp/fairness_matched_age_xai_{os.getpid()}_{uuid.uuid4().hex}")
local_root.mkdir(parents=True, exist_ok=True)

def stable_key(value): return hashlib.sha256(str(value).encode()).hexdigest()[:20]
def digest(path):
    hasher = hashlib.sha256(); total = 0
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            total += len(chunk); hasher.update(chunk)
    return total, hasher.hexdigest()
def publish(local_path, volume_path, retries=4):
    local_path, volume_path = Path(local_path), Path(volume_path); volume_path.parent.mkdir(parents=True, exist_ok=True)
    expected = digest(local_path)
    if volume_path.is_file():
        try:
            if digest(volume_path) == expected: return volume_path
        except OSError: pass
    last = None
    for attempt in range(retries):
        partial = volume_path.with_name(f".{volume_path.name}.{uuid.uuid4().hex}.partial")
        try:
            with local_path.open("rb") as source, partial.open("wb") as destination:
                shutil.copyfileobj(source, destination, 8 * 1024 * 1024); destination.flush(); os.fsync(destination.fileno())
            if digest(partial) != expected: raise OSError("partial digest mismatch")
            os.replace(partial, volume_path)
            if digest(volume_path) != expected: raise OSError("published digest mismatch")
            return volume_path
        except Exception as error:
            last = error; partial.unlink(missing_ok=True); time.sleep(2 ** attempt)
    raise OSError(f"Could not publish {volume_path}") from last

selection["image_key"] = selection["image_path"].map(stable_key)
selection["record_key"] = selection.apply(lambda row: stable_key(f"{row['comparison_group']}|{row['participant_id']}|{row['image_path']}"), axis=1)
ordered = selection.sort_values(["comparison_group", "population_role", "participant_id"], kind="stable").reset_index(drop=True)
expected_pairs = {
    (row.record_key, model_name)
    for row in ordered.itertuples(index=False)
    for model_name in relevant_model_names(row.comparison_group)
}
n_batches = math.ceil(len(ordered) / segmentation_batch_size)
completed_manifests = []; new_batches = 0
for batch_number, start in enumerate(range(0, len(ordered), segmentation_batch_size), 1):
    stop = min(start + segmentation_batch_size, len(ordered)); batch = ordered.iloc[start:stop].copy()
    batch_dir = batch_root / f"batch_{start:07d}_{stop:07d}"
    manifest_path = batch_dir / "matched_age_head_anatomic_explainability.parquet"
    expected_batch_pairs = {(row.record_key, name) for row in batch.itertuples(index=False) for name in relevant_model_names(row.comparison_group)}
    if resume_batches and manifest_path.is_file():
        existing = pd.read_parquet(manifest_path)
        existing_pairs = set(zip(existing["record_key"].astype(str), existing["model_name"].astype(str))) if {"record_key", "model_name"}.issubset(existing.columns) else set()
        paths_ready = {"attribution_npz_path", "mask_path"}.issubset(existing.columns) and all(Path(value).is_file() and Path(value).stat().st_size > 0 for column in ("attribution_npz_path", "mask_path") for value in existing[column].dropna().astype(str).unique())
        if existing_pairs == expected_batch_pairs and paths_ready:
            completed_manifests.append(manifest_path); print(f"[xai {batch_number}/{n_batches}] resumed {len(batch)} records", flush=True); continue
    print(f"[xai {batch_number}/{n_batches}] records {start:,}:{stop:,}", flush=True)
    processed = [preprocess_fundus(path, quality_config) for path in batch["image_path"]]
    rgb_images = [np.asarray(item.image.convert("RGB")) for item in processed]
    coordinates = np.asarray(landmark_model.predict(rgb_images), dtype=float).reshape(-1, 4)
    vessels = np.asarray(fit.ensemble_predict_segmentation(vessel_ensemble, rgb_images, device=device, size=(512, 512), threshold=vessel_threshold), dtype=float)
    if vessels.ndim == 2: vessels = vessels[None, ...]
    rows = []; batch_dir.mkdir(parents=True, exist_ok=True)
    for position, (_, record) in enumerate(batch.iterrows()):
        rgb = rgb_images[position]; vessel = vessels[position]
        if vessel.shape != rgb.shape[:2]: vessel = np.asarray(Image.fromarray(vessel.astype(np.float32)).resize((rgb.shape[1], rgb.shape[0]), Image.Resampling.BILINEAR))
        proxies = fundus_physiology_proxies(rgb)
        masks, anatomy_metadata = build_anatomic_masks(rgb.shape[:2], coordinates[position], vessel, proxies["retina"], optic_disc_radius_scale=optic_disc_radius_scale, fovea_radius_scale=fovea_radius_scale, peripapillary_multiplier=peripapillary_multiplier)
        names = relevant_model_names(record["comparison_group"]); heads = {name: linear_heads[name] for name in names}
        contributions = exact_multihead_patch_contributions(retfound_model, heads, record["image_path"], retfound_device, quality_config)
        if max(item["reconstruction_error"] for item in contributions.values()) > 1e-4: raise RuntimeError("Exact contribution reconstruction failed")
        key = str(record["record_key"]); array_keys = {name: f"head_{index:03d}" for index, name in enumerate(names)}
        local_maps = local_root / f"{key}_age_heads.npz"; np.savez_compressed(local_maps, **{array_keys[name]: contributions[name]["variable_grid"] for name in names})
        maps_path = artifact_root / "attributions" / local_maps.name; publish(local_maps, maps_path); local_maps.unlink(missing_ok=True)
        local_masks = local_root / f"{key}_anatomic_masks.npz"; np.savez_compressed(local_masks, **{name: value.astype(np.uint8) for name, value in masks.items()})
        mask_path = artifact_root / "masks" / local_masks.name; publish(local_masks, mask_path); local_masks.unlink(missing_ok=True)
        role_lookup = {"global": "global", comparison_head_lookup[record["comparison_group"]]["target"]: "target", comparison_head_lookup[record["comparison_group"]]["matched_white"]: "matched_white"}
        for model_name in names:
            rows.append({
                "record_key": key, "image_key": record["image_key"], "image_path": record["image_path"],
                "participant_id": str(record["participant_id"]), "comparison_group": str(record["comparison_group"]),
                "population_role": str(record["population_role"]), "racial_background": str(record["racial_background"]),
                "age": float(record["age"]), "model_name": model_name, "model_role": role_lookup[model_name],
                "prediction_years": contributions[model_name]["prediction_from_feature"],
                "retinal_age_gap": contributions[model_name]["prediction_from_feature"] - float(record["age"]),
                "reconstruction_error": contributions[model_name]["reconstruction_error"],
                "attribution_npz_path": str(maps_path), "attribution_array_key": array_keys[model_name],
                "mask_path": str(mask_path), "fit_source_commit": FIT_SOURCE_COMMIT,
                **anatomy_metadata, **attribution_region_metrics(contributions[model_name]["variable_grid"], proxies["retina"], masks),
            })
    manifest = pd.DataFrame(rows); local_manifest = local_root / f"batch_{start:07d}_{stop:07d}.parquet"
    write_frame(manifest, local_manifest); publish(local_manifest, manifest_path); local_manifest.unlink(missing_ok=True)
    completed_manifests.append(manifest_path); new_batches += 1
    print(f"[xai {batch_number}/{n_batches}] saved {len(manifest)} model-record rows", flush=True)
    del processed, rgb_images, coordinates, vessels, rows, manifest; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if max_new_batches_per_run and new_batches >= max_new_batches_per_run and batch_number < n_batches:
        dbutils.notebook.exit(json.dumps({"status": "checkpointed_incomplete", "completed_batches": len(completed_manifests), "total_batches": n_batches, "next_action": "Rerun notebook 02"}, indent=2))

all_rows = pd.concat([pd.read_parquet(path) for path in completed_manifests], ignore_index=True)
actual_pairs = set(zip(all_rows["record_key"].astype(str), all_rows["model_name"].astype(str)))
if actual_pairs != expected_pairs or all_rows.duplicated(["record_key", "model_name"]).any(): raise ValueError("Completed batches do not exactly cover the comparison-record x model plan")
consolidated_path = private_root / "matched_age_head_anatomic_explainability_private.parquet"
local_consolidated = local_root / consolidated_path.name; write_frame(all_rows, local_consolidated); publish(local_consolidated, consolidated_path); local_consolidated.unlink(missing_ok=True)
display(all_rows.groupby(["comparison_group", "population_role", "model_role"]).agg(participants=("participant_id", "nunique"), anatomy_valid_fraction=("anatomy_valid", "mean")))


## 6. Paired model-map and matched-population inference

Two estimands are separated:

1. On identical target images, does the target head spatially differ from the
   matched-White or global head? These are participant-paired sign-flip tests.
2. Under each population's own head, do target participants differ spatially
   from their matched White cohort? These are participant-level permutation
   tests within each comparison.


In [ ]:
primary_metrics = [
    "optic_disc_roi_positive_enrichment", "peripapillary_annulus_positive_enrichment",
    "optic_disc_plus_peripapillary_positive_enrichment", "fovea_roi_positive_enrichment",
    "vessels_positive_enrichment", "vessels_elsewhere_positive_enrichment", "other_retina_positive_enrichment",
    "optic_disc_roi_absolute_enrichment", "peripapillary_annulus_absolute_enrichment",
    "fovea_roi_absolute_enrichment", "vessels_absolute_enrichment", "other_retina_absolute_enrichment",
]
valid = all_rows[all_rows["anatomy_valid"].astype(bool)].copy()
available_metrics = [name for name in primary_metrics if name in valid.columns and np.isfinite(pd.to_numeric(valid[name], errors="coerce")).all()]
if not available_metrics: raise ValueError("No complete anatomic metrics remain")
target_rows = valid[valid["population_role"].astype(str).eq("target")].copy()
global_reference = paired_model_inference(target_rows, available_metrics, model_column="model_role", reference_model="global", target_group_column="comparison_group", permutations=permutations, bootstrap_repetitions=bootstrap_repetitions)
global_reference = global_reference[global_reference["subgroup_model"].eq("target")].copy()
white_reference = paired_model_inference(target_rows, available_metrics, model_column="model_role", reference_model="matched_white", target_group_column="comparison_group", permutations=permutations, bootstrap_repetitions=bootstrap_repetitions)
white_reference = white_reference[white_reference["subgroup_model"].eq("target")].copy()
write_frame(global_reference, statistics_root / "target_head_vs_global_on_target_images.csv")
write_frame(white_reference, statistics_root / "target_head_vs_matched_white_head_on_target_images.csv")

population_inference_frames = []
for group_index, group in enumerate(comparison_groups):
    diagonal = valid[
        (valid["comparison_group"].astype(str) == group)
        & (((valid["population_role"] == "target") & (valid["model_role"] == "target"))
           | ((valid["population_role"] == "matched_white") & (valid["model_role"] == "matched_white")))
    ].copy()
    diagonal["population_label"] = (diagonal["population_role"] == "target").astype(int)
    diagonal = diagonal.drop_duplicates("participant_id")
    if set(diagonal["population_label"]) != {0, 1}: continue
    inference = participant_permutation_inference(
        diagonal[["participant_id", "population_label", *available_metrics]], available_metrics,
        label_column="population_label", permutations=permutations,
        bootstrap_repetitions=bootstrap_repetitions, random_state=20260822 + group_index,
    ).rename(columns={"glaucoma_minus_healthy": "target_minus_matched_white"})
    inference["comparison_group"] = group; population_inference_frames.append(inference)
population_inference = pd.concat(population_inference_frames, ignore_index=True)
write_frame(population_inference, statistics_root / "target_vs_matched_white_population_anatomic_inference.csv")
display(white_reference.sort_values(["target_racial_background", "signflip_p_max_t"]).round(5))
display(population_inference.sort_values(["comparison_group", "permutation_p_max_t"]).round(5))


## 7. Registered average heatmaps and anatomy overlays

Every image is mapped to a common disc–fovea axis (disc left, fovea right). Heatmaps are scaled within image by the 99th percentile of absolute attribution before averaging, so a few extreme-magnitude images cannot dominate spatial pattern.


In [ ]:
def load_registered(row, size=256):
    processed = preprocess_fundus(row["image_path"], quality_config); rgb = np.asarray(processed.image.convert("RGB"), dtype=np.uint8)
    with np.load(row["mask_path"]) as archive: masks = {name: np.asarray(archive[name], dtype=bool) for name in archive.files}
    with np.load(row["attribution_npz_path"]) as archive: grid = np.asarray(archive[row["attribution_array_key"]], dtype=float)
    resized = ndimage.zoom(grid, (rgb.shape[0]/grid.shape[0], rgb.shape[1]/grid.shape[1]), order=1)[:rgb.shape[0], :rgb.shape[1]]
    retina = np.logical_or.reduce(list(masks.values())); scale = max(float(np.quantile(np.abs(resized[retina]), .99)), 1e-8); resized = np.clip(resized/scale, -1, 1)
    matrix = disc_fovea_affine_matrix((row["fovea_x_px"], row["fovea_y_px"], row["optic_disc_x_px"], row["optic_disc_y_px"]), output_size=size)
    def warp(array, interpolation): return cv2.warpAffine(array, matrix, (size,size), flags=interpolation, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return {"rgb": warp(rgb, cv2.INTER_LINEAR), "heat": warp(resized.astype(np.float32), cv2.INTER_LINEAR), "retina": warp(retina.astype(np.uint8), cv2.INTER_NEAREST)>0, "masks": {name: warp(mask.astype(np.uint8), cv2.INTER_NEAREST)>0 for name,mask in masks.items()}}

plot_rows = valid[
    ((valid["population_role"] == "target") & valid["model_role"].isin(["target", "global"]))
    | ((valid["population_role"] == "matched_white") & (valid["model_role"] == "matched_white"))
].copy()
plot_rows["panel"] = np.select(
    [(plot_rows["population_role"] == "target") & (plot_rows["model_role"] == "target"),
     (plot_rows["population_role"] == "target") & (plot_rows["model_role"] == "global")],
    ["target_own_head", "target_global_head"], default="white_own_head",
)
accumulators = {}
for key in {(str(row.comparison_group), str(row.panel)) for row in plot_rows.itertuples()}:
    accumulators[key] = {"n":0,"coverage":np.zeros((registration_size,registration_size)),"heat":np.zeros((registration_size,registration_size)),"rgb":np.zeros((registration_size,registration_size,3)),"vessels":np.zeros((registration_size,registration_size)),"disc":np.zeros((registration_size,registration_size)),"fovea":np.zeros((registration_size,registration_size))}
for index, (_, row) in enumerate(plot_rows.iterrows(), 1):
    arrays = load_registered(row, registration_size); acc = accumulators[(str(row["comparison_group"]), str(row["panel"]))]; retina = arrays["retina"].astype(float)
    acc["n"] += 1; acc["coverage"] += retina; acc["heat"] += arrays["heat"]*retina; acc["rgb"] += arrays["rgb"].astype(float)/255*retina[...,None]
    acc["vessels"] += arrays["masks"]["vessels"].astype(float); acc["disc"] += arrays["masks"]["optic_disc_roi"].astype(float); acc["fovea"] += arrays["masks"]["fovea_roi"].astype(float)
    if index % 100 == 0: print(f"[registered averages] {index:,}/{len(plot_rows):,}", flush=True)
averages = {}
for key,acc in accumulators.items():
    coverage=np.maximum(acc["coverage"],1); averages[key]={"n":acc["n"],"heat":acc["heat"]/coverage,"rgb":acc["rgb"]/coverage[...,None],"vessels":acc["vessels"]/coverage,"disc":acc["disc"]/coverage,"fovea":acc["fovea"]/coverage}

fig, axes = plt.subplots(len(comparison_groups), 5, figsize=(17,3.2*len(comparison_groups)), squeeze=False)
for row_index, group in enumerate(comparison_groups):
    target=averages[(group,"target_own_head")]; white=averages[(group,"white_own_head")]; global_map=averages[(group,"target_global_head")]
    axes[row_index,0].imshow(target["rgb"]); axes[row_index,0].imshow(target["vessels"],cmap="Reds",alpha=.4); axes[row_index,0].contour(target["disc"],[.5],colors="cyan"); axes[row_index,0].contour(target["fovea"],[.5],colors="lime"); axes[row_index,0].set_title(f"{group} anatomy (n={target['n']})")
    axes[row_index,1].imshow(target["heat"],cmap="coolwarm",vmin=-.35,vmax=.35); axes[row_index,1].set_title("Target own head")
    axes[row_index,2].imshow(white["heat"],cmap="coolwarm",vmin=-.35,vmax=.35); axes[row_index,2].set_title(f"Matched White own head (n={white['n']})")
    axes[row_index,3].imshow(global_map["heat"],cmap="coolwarm",vmin=-.35,vmax=.35); axes[row_index,3].set_title("Global head on target")
    difference=target["heat"]-white["heat"]; limit=max(float(np.quantile(np.abs(difference),.99)),.02); axes[row_index,4].imshow(difference,cmap="PuOr",vmin=-limit,vmax=limit); axes[row_index,4].set_title("Target minus matched White")
    for axis in axes[row_index]: axis.axis("off")
fig.suptitle("All-available matched CLSA age-head attribution registered to the disc–fovea axis",y=1.002); fig.tight_layout()
fig.savefig(figure_root/"registered_all_available_target_vs_matched_white_heatmaps.png",dpi=220,bbox_inches="tight"); display(fig); plt.close(fig)


### 7A. Deidentified participant-level review panels

Examples are selected at evenly spaced global age-gap ranks within each released group—not by heatmap appearance. Each row shows the same held-out image under the global and corresponding own-group head.


In [ ]:
representatives=[]
for group, group_frame in target_rows[target_rows["model_role"]=="global"].groupby("comparison_group",sort=True):
    ordered_group=group_frame.sort_values(["retinal_age_gap","record_key"],kind="stable").reset_index(drop=True)
    positions=np.linspace(0,len(ordered_group)-1,min(review_examples_per_group,len(ordered_group))).round().astype(int); representatives.append(ordered_group.iloc[np.unique(positions)])
representatives=pd.concat(representatives,ignore_index=True)
fig,axes=plt.subplots(len(representatives),5,figsize=(17,3.1*len(representatives)),squeeze=False)
for row_index,(_,global_row) in enumerate(representatives.iterrows()):
    same=target_rows[(target_rows["record_key"]==global_row["record_key"])]
    target_row=same[same["model_role"]=="target"].iloc[0]; white_row=same[same["model_role"]=="matched_white"].iloc[0]
    global_arrays=load_registered(global_row,registration_size); target_arrays=load_registered(target_row,registration_size); white_arrays=load_registered(white_row,registration_size)
    axes[row_index,0].imshow(global_arrays["rgb"]); axes[row_index,0].set_title(f"{global_row['comparison_group']} | age={global_row['age']:.1f}")
    axes[row_index,1].imshow(global_arrays["rgb"]); axes[row_index,1].imshow(global_arrays["masks"]["vessels"],cmap="Reds",alpha=.55)
    for name,color in (("optic_disc_roi","cyan"),("peripapillary_annulus","yellow"),("fovea_roi","lime")): axes[row_index,1].contour(global_arrays["masks"][name],[.5],colors=color,linewidths=.8)
    axes[row_index,1].set_title("Registered anatomy")
    for column,(label,arrays,row) in enumerate((("Global",global_arrays,global_row),("Target head",target_arrays,target_row),("Matched-White head",white_arrays,white_row)),2):
        axes[row_index,column].imshow(arrays["rgb"]); axes[row_index,column].imshow(arrays["heat"],cmap="coolwarm",vmin=-1,vmax=1,alpha=.52); axes[row_index,column].set_title(f"{label} | gap={row['retinal_age_gap']:+.1f} y")
    for axis in axes[row_index]: axis.axis("off")
fig.suptitle("Same held-out target image under global, target, and matched-White heads",y=1.002); fig.tight_layout()
fig.savefig(figure_root/"representative_three_head_target_image_overlays.png",dpi=220,bbox_inches="tight"); display(fig); plt.close(fig)


## 8. Interpretation contract and run summary

A region is described as preferentially weighted only when its attribution enrichment exceeds retinal area and the paired participant-level analysis supports the contrast after multiplicity control. Attribution is model explanation, not proof of a causal biological mechanism. Between-group differences may still reflect residual image, socioeconomic, clinical, or measurement differences.


In [ ]:
significant_paired = white_reference[white_reference["signflip_p_max_t"] < .05].to_dict("records")
significant_population = population_inference[population_inference["permutation_p_max_t"] < .05].to_dict("records")
summary={
    "analysis":"CLSA_all_available_matched_age_head_anatomic_fairness",
    "n_comparison_records":int(len(ordered)), "n_unique_participants_across_comparisons":int(ordered["participant_id"].nunique()),
    "comparison_groups":comparison_groups, "n_age_heads":len(linear_heads),
    "anatomy_valid_fraction":float(all_rows.drop_duplicates("record_key")["anatomy_valid"].mean()),
    "fit_source_commit":FIT_SOURCE_COMMIT, "retfound_checkpoint":str(resolved_checkpoint),
    "significant_target_head_vs_white_head_regions_max_t":significant_paired,
    "significant_target_vs_matched_white_population_regions_max_t":significant_population,
    "definitions":{"vasculature":"FR-U-Net pixel segmentation","optic_disc_and_fovea":"circular ROIs around localized centers; not segmentations","latent_coefficients":"unnamed RETFound dimensions"},
    "limitations":["Attribution is model explanation, not causal physiology.","Released racial/cultural background is self-report, not genetic ancestry.","All-available subgroup heads differ in training N; retain the equal-N analysis as sensitivity.","Final frozen heads are explained; cross-fitted predictions in notebook 01 remain the performance estimand."],
    "outputs":{"manifest":str(consolidated_path),"paired_head_inference":str(statistics_root/"target_head_vs_matched_white_head_on_target_images.csv"),"population_inference":str(statistics_root/"target_vs_matched_white_population_anatomic_inference.csv"),"heatmaps":str(figure_root/"registered_all_available_target_vs_matched_white_heatmaps.png")},
}
write_json(summary,output_root/"ALL_AVAILABLE_MATCHED_ANATOMIC_EXPLAINABILITY_SUMMARY.json"); print(json.dumps(summary,indent=2,default=str))
os.environ.pop("HF_TOKEN",None)
try: dbutils.widgets.remove("hf_token")
except Exception: pass
print("Notebook 02 complete")
